# Per-Label Saliency Heatmaps — Non-MTL Models

Produces **7 figures per model** (one per class label). Each figure has **4 column groups**:

| Column group | Sample filter | Heatmap | Bottom summary row |
|---|---|---|---|
| **A-Raw** | `y_true==c AND y_pred==c` | latent `\|∂Z/∂X\|` raw | Σ\|∂Z/∂X\| collapsed to 1D (raw) |
| **A-Norm** | same | latent `\|∂Z/∂X\|` row-normalised | Σ\|∂Z/∂X\| collapsed to 1D (row-norm) |
| **B-Raw** | `y_true==c` (or all, toggle) | same latent dims, B-mask, raw | `\|d(cls_c)/dX\|` direct gradient |
| **B-Norm** | same | same latent dims, B-mask, normalised | `\|d(cls_c)/dX\|` direct gradient |

Each column contains (top → bottom):
1. **Curve plot** (mean ± std of filtered samples)
2. **CNN latent heatmap** (n_dims × T)
3. **GRU/Trans latent heatmap** (dual models only)
4. **1-D summary line** overlaid on the mean curve

**View A summary** (blue): Σ|∂Z/∂X| averaged over all latent dims — the temporal shape of the pooled latent attribution.  
**View B summary** (orange): |d(cls_c)/dX| direct input-space gradient — what time points drive the class-c logit directly.


In [1]:
import os, sys
sys.path.insert(0, '/vol/bitbucket/gk225/POC_DDM/gk_code/main')
from pathlib import Path
import numpy as np
import config

# ── Paths ──────────────────────────────────────────────────────────────────
MAIN_DIR     = Path('/vol/bitbucket/gk225/POC_DDM/gk_code/main')
UTILS_DIR    = MAIN_DIR / 'utils'
UTILS_MT_DIR = UTILS_DIR / 'model_training'
EXP_NAME     = 'D20260731_E00_C00_F4500KHz_U_DDM_01_04'
EXP_PATH     = Path('/vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract', EXP_NAME)
SAVE_DIR     = EXP_PATH / 'model_interpretation' / 'per_label_saliency'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Parameters ─────────────────────────────────────────────────────────────
CURVE_IDX    = 0          # 0 = ori_curve, 1 = ori_curve_avg, 2 = wavelet
N_DIMS       = 25         # top N latent dims shown in View A
SAMPLE_SCOPE = 'class'    # View B averaging: 'class' (default) | 'all'
BATCH_N      = 512        # samples to draw for gradient computation
RANDOM_SEED  = 42

# LabelEncoder alphabetical order for this dataset
LABEL_MAPPING = config.get_label_mappings(EXP_PATH)[EXP_NAME]
LABEL_NAMES = list(np.unique(list((LABEL_MAPPING.values()))))
N_CLASSES   = len(LABEL_NAMES)


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



In [2]:
import sys, importlib.util, joblib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Patch
from sklearn.preprocessing import LabelEncoder

# ── sys.path so 07_attribution_vis_all.py can resolve its own imports ───────
for p in [str(MAIN_DIR), str(UTILS_DIR), str(UTILS_MT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Register custom Keras serialisable classes before any model load ────────
import model_utils_gated        # noqa — registers _SumPool1D, _OneMinus
import model_utils_mtl          # noqa — registers MTLModel
import model_utils_supcon       # noqa — registers SupConModel variants
import model_utils_rcfd         # noqa — registers RCFDModel variants
import model_utils_arch_poc_st  # noqa — registers ArchPocSTCCGD* variants

# ── Import utility functions from 07_attribution_vis_all.py via importlib ──
_spec = importlib.util.spec_from_file_location(
    'xai07', MAIN_DIR / '07_attribution_vis_all.py'
)
xai07 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(xai07)

extract_xai_artifacts              = xai07.extract_xai_artifacts
normalize_heatmap                  = xai07.normalize_heatmap
find_bidirectional_recurrent_layer = xai07.find_bidirectional_recurrent_layer
find_flatten_layer                 = xai07.find_flatten_layer
find_transformer_block_output      = xai07.find_transformer_block_output

print('Imports OK  |  TF', tf.__version__)

Imports OK  |  TF 2.16.2


In [3]:
# ── Load dataset ───────────────────────────────────────────────────────────
data       = joblib.load(EXP_PATH / 'curve_for_training.joblib')
timestamps = np.array(data['timestamps'])                        # (45,)
X_all      = data['dataset'][CURVE_IDX].astype(np.float32)[..., None]  # (N,45,1)

data['Y_well'] = [LABEL_MAPPING.get(w, w) for w in data['Y_well']]
enc   = LabelEncoder()
y_all = enc.fit_transform(data['Y_well'])

assert list(enc.classes_) == LABEL_NAMES, \
    f'Label order mismatch: {list(enc.classes_)}'

print(f'X: {X_all.shape}  |  T={len(timestamps)} pts  |  {N_CLASSES} classes')
for i, name in enumerate(LABEL_NAMES):
    print(f'  [{i}] {name}: {(y_all==i).sum()} samples')

# ── Draw a reproducible batch for gradient computation ─────────────────────
rng     = np.random.default_rng(RANDOM_SEED)
idx     = rng.choice(len(X_all), size=min(BATCH_N, len(X_all)), replace=False)
X_batch = X_all[idx]
y_batch = y_all[idx]
print(f'\nBatch drawn: {X_batch.shape}  (seed={RANDOM_SEED})')

X: (18094, 405, 1)  |  T=405 pts  |  7 classes
  [0] Cov: 1793 samples
  [1] Hadv: 3624 samples
  [2] IAV: 5674 samples
  [3] IBV: 1795 samples
  [4] Kp: 1838 samples
  [5] NC: 1714 samples
  [6] PC: 1656 samples

Batch drawn: (512, 405, 1)  (seed=42)


In [4]:
# ── Load non-MTL models ────────────────────────────────────────────────────
#
# Excluded:
#   _mtl  — multi-task models (cls+reg), no latent decomp stored in artifacts
#   rcfd  — treated as MTL in 07_attribution_vis_all (also has regression conditioning)
#
MODEL_DIR    = EXP_PATH / 'model_interpretation'
all_keras    = sorted(MODEL_DIR.glob('*_None_ori_curve_model.keras'))
non_mtl_files = [
    f for f in all_keras
    if '_mtl' not in f.stem and 'rcfd' not in f.stem
]

def _model_name(path: Path) -> str:
    return path.stem.replace('_None_ori_curve_model', '')

print(f'Found {len(non_mtl_files)} non-MTL models:')
for f in non_mtl_files:
    print(' ', _model_name(f))

models = {}
for f in non_mtl_files:
    name = _model_name(f)
    try:
        models[name] = tf.keras.models.load_model(f)
        print(f'  [OK] {name}')
    except Exception as e:
        print(f'  [!] {name}: {e}')

print(f'\nLoaded {len(models)} models.')

Found 9 non-MTL models:
  cnn_gru_dual
  cnn_gru_dual_attn_recon
  cnn_gru_dual_attn_recon_supcon2
  cnn_gru_dual_attn_recon_supcon
  cnn_gru_dual_cosine_recon
  cnn_gru_dual_cosine_recon_supcon2
  cnn_gru_dual_cosine_recon_supcon
  cnn_gru_dual_supcon2
  cnn_gru_dual_supcon
  [OK] cnn_gru_dual
  [OK] cnn_gru_dual_attn_recon
  [OK] cnn_gru_dual_attn_recon_supcon2
  [OK] cnn_gru_dual_attn_recon_supcon
  [OK] cnn_gru_dual_cosine_recon
  [OK] cnn_gru_dual_cosine_recon_supcon2
  [OK] cnn_gru_dual_cosine_recon_supcon
  [OK] cnn_gru_dual_supcon2
  [OK] cnn_gru_dual_supcon

Loaded 9 models.


In [5]:
# ── Extract XAI artifacts (latent saliency maps, preserve batch dim) ───────
#
# extract_xai_artifacts also needs X_man_batch (manual features for LF models).
# No LF models here — pass a zero dummy. The function skips X_man for dual/base paths.
#
X_man_dummy = np.zeros((len(X_batch), 10), dtype=np.float32)

print('Computing XAI artifacts (View A latent saliency)...')
artifacts = extract_xai_artifacts(models, X_batch, X_man_dummy)

# Filter to non-MTL artifact types
non_mtl_artifacts = {
    k: v for k, v in artifacts.items()
    if v.get('is_type') in ('base', 'dual')
}
print('\nArtifact types:')
for k, v in non_mtl_artifacts.items():
    print(f'  {k}: {v["is_type"]}  '
          f'  raw_saliency_curve dims={len(v["raw_saliency_curve"])}')

Computing XAI artifacts (View A latent saliency)...
    [+] Computing Gradients: cnn_gru_dual
    [+] Computing Gradients: cnn_gru_dual_attn_recon
    [+] Computing Gradients: cnn_gru_dual_attn_recon_supcon2
    [+] Computing Gradients: cnn_gru_dual_attn_recon_supcon
    [+] Computing Gradients: cnn_gru_dual_cosine_recon
    [+] Computing Gradients: cnn_gru_dual_cosine_recon_supcon2
    [+] Computing Gradients: cnn_gru_dual_cosine_recon_supcon
    [+] Computing Gradients: cnn_gru_dual_supcon2
    [+] Computing Gradients: cnn_gru_dual_supcon

Artifact types:
  cnn_gru_dual: dual    raw_saliency_curve dims=100
  cnn_gru_dual_attn_recon: base    raw_saliency_curve dims=1
  cnn_gru_dual_attn_recon_supcon2: base    raw_saliency_curve dims=1
  cnn_gru_dual_attn_recon_supcon: base    raw_saliency_curve dims=1
  cnn_gru_dual_cosine_recon: base    raw_saliency_curve dims=32
  cnn_gru_dual_cosine_recon_supcon2: dual    raw_saliency_curve dims=100
  cnn_gru_dual_cosine_recon_supcon: dual    raw_s

In [6]:
# ── Helper: View B direct input gradient ──────────────────────────────────

def _to_model_input(model: tf.keras.Model, X: np.ndarray) -> np.ndarray:
    """Build the correct input array for model, handling neighbor-stack architectures.

    Standard models expect (N, T, 1). Models with neighbor_stack_input expect
    (N, k+1, T) — built by repeating each curve k+1 times along a new axis.
    """
    inp = model.input
    if (not isinstance(inp, (list, tuple))
            and hasattr(inp, 'name')
            and 'neighbor_stack' in inp.name):
        k_plus_1 = inp.shape[1]
        center   = X[:, :, 0]                                      # (N, T)
        return np.stack([center] * k_plus_1, axis=1).astype(np.float32)  # (N, k+1, T)
    return X


def compute_class_direct_saliency(
    model: tf.keras.Model,
    X_class: np.ndarray,   # (N_c, T, 1) — already filtered to the scope
    class_idx: int,
) -> np.ndarray:           # (T,) mean |d(cls[:,c])/d(X)|
    """
    Differentiates the class-c output logit w.r.t. each input sample.

    Uses reduce_sum (not reduce_mean) so that, for architectures without
    cross-sample coupling (no BatchNorm in inference mode), gradient[i] equals
    d(cls[i, c]) / d(X[i]) exactly — samples are independent and the sum
    gradient decomposes sample-wise.
    """
    x_var = tf.Variable(_to_model_input(model, X_class))
    with tf.GradientTape() as tape:
        outs   = model(x_var, training=False)
        cls_t  = outs[0] if isinstance(outs, (list, tuple)) else outs
        target = tf.reduce_sum(cls_t[:, class_idx])
    grad = tape.gradient(target, x_var).numpy()     # (N, T, 1) or (N, k+1, T)
    if grad.ndim == 3 and grad.shape[-1] != 1:      # neighbor-stack: (N, k+1, T)
        return np.abs(grad).mean(axis=(0, 1))        # (T,)
    return np.abs(grad[:, :, 0]).mean(axis=0)        # (T,)


def _get_predictions(model: tf.keras.Model, X: np.ndarray,
                     batch_size: int = 256) -> np.ndarray:
    """Integer class predictions, handles multi-output and neighbor-stack models."""
    chunks = []
    for i in range(0, len(X), batch_size):
        x_in   = _to_model_input(model, X[i:i + batch_size])
        out    = model(x_in, training=False)
        logits = out[0] if isinstance(out, (list, tuple)) else out
        chunks.append(logits.numpy())
    return np.argmax(np.concatenate(chunks, axis=0), axis=1)

In [ ]:
# ── Main plot function ─────────────────────────────────────────────────────

def plot_per_label_saliency_heatmap(
    art: dict,
    model: tf.keras.Model,
    X_batch: np.ndarray,    # (N, T, 1)
    y_batch: np.ndarray,    # (N,) int
    y_pred:  np.ndarray,    # (N,) int
    class_idx:   int,
    class_name:  str,
    timestamps:  np.ndarray,
    n_dims:       int  = 25,
    sample_scope: str  = 'class',   # 'class' | 'all'
    save_path = None,
) -> None:
    """
    1 figure per label. 4 column groups: A-Raw | A-Norm | B-Raw | B-Norm.
    Each column: curve (top, taller) -> CNN heatmap -> GRU heatmap (dual) -> 1D summary.
    View A: correctly-classified samples; summary = collapsed latent saliency Sum|dZ/dX|.
    View B: class-c samples (or all); summary = direct |d(cls_c)/dX|.
    """
    is_dual = art['is_type'] == 'dual'
    T       = len(timestamps)
    t_step  = (timestamps[-1] - timestamps[0]) / (T - 1) if T > 1 else 1.0
    t_start = timestamps[0]  - t_step / 2
    t_end   = timestamps[-1] + t_step / 2

    # ── Masks ──────────────────────────────────────────────────────────────
    mask_a    = (y_batch == class_idx) & (y_pred == class_idx)
    mask_b    = (y_batch == class_idx) if sample_scope == 'class' \
                else np.ones(len(y_batch), dtype=bool)
    n_a       = int(mask_a.sum())
    n_b       = int(mask_b.sum())
    n_total_c = int((y_batch == class_idx).sum())

    if n_a == 0:
        print(f'  [skip] {class_name}: no correctly-classified samples in batch')
        return
    if n_b == 0:
        print(f'  [skip] {class_name}: no View B samples')
        return

    # ── Heatmaps ───────────────────────────────────────────────────────────
    n_cnn = min(n_dims, len(art['raw_saliency_curve']))
    hm_a_cnn_raw  = np.array([m[mask_a].mean(0) for m in art['raw_saliency_curve'][:n_cnn]])
    hm_a_cnn_norm = normalize_heatmap(hm_a_cnn_raw, method='row')
    hm_b_cnn_raw  = np.array([m[mask_b].mean(0) for m in art['raw_saliency_curve'][:n_cnn]])
    hm_b_cnn_norm = normalize_heatmap(hm_b_cnn_raw, method='row')

    if is_dual:
        n_rnn   = min(n_dims, len(art['raw_saliency_rnn']))
        _branch = 'Trans' if any('trans' in l.name for l in model.layers) else 'GRU'
        hm_a_rnn_raw  = np.array([m[mask_a].mean(0) for m in art['raw_saliency_rnn'][:n_rnn]])
        hm_a_rnn_norm = normalize_heatmap(hm_a_rnn_raw, method='row')
        hm_b_rnn_raw  = np.array([m[mask_b].mean(0) for m in art['raw_saliency_rnn'][:n_rnn]])
        hm_b_rnn_norm = normalize_heatmap(hm_b_rnn_raw, method='row')

    # ── 1-D summary signals ────────────────────────────────────────────────
    def _norm1d(x):
        return (x - x.min()) / (x.ptp() + 1e-12)

    # View A: collapse heatmap over latent-dim axis -> temporal summary
    _a_raw_sum = hm_a_cnn_raw.mean(0)
    _a_nrm_sum = hm_a_cnn_norm.mean(0)
    if is_dual:
        _a_raw_sum = _a_raw_sum + hm_a_rnn_raw.mean(0)
        _a_nrm_sum = _a_nrm_sum + hm_a_rnn_norm.mean(0)
    sal_a_raw_1d = _norm1d(_a_raw_sum)
    sal_a_nrm_1d = _norm1d(_a_nrm_sum)

    # View B: direct class gradient |d(cls_c)/dX| -> (T,)
    sal_b_1d = _norm1d(
        compute_class_direct_saliency(model, X_batch[mask_b], class_idx)
    )

    # ── Mean curves ────────────────────────────────────────────────────────
    mean_a = X_batch[mask_a, :, 0].mean(0)
    std_a  = X_batch[mask_a, :, 0].std(0)
    mean_b = X_batch[mask_b, :, 0].mean(0)
    std_b  = X_batch[mask_b, :, 0].std(0)

    # ── Figure / GridSpec ──────────────────────────────────────────────────
    # outer [View A | View B], each inner [Raw col | Norm col]
    n_hm   = 2 if is_dual else 1
    hr     = [1.8] + [3.2] * n_hm + [1.3]
    n_rows = 1 + n_hm + 1
    fig    = plt.figure(figsize=(28 if is_dual else 22, 17 if is_dual else 12),
                        facecolor='white')
    outer  = gridspec.GridSpec(1, 2, wspace=0.42, figure=fig)
    kw     = dict(height_ratios=hr, hspace=0.20, wspace=0.24)
    gs_a   = gridspec.GridSpecFromSubplotSpec(n_rows, 2, subplot_spec=outer[0], **kw)
    gs_b   = gridspec.GridSpecFromSubplotSpec(n_rows, 2, subplot_spec=outer[1], **kw)

    # ── Helper: curve row ──────────────────────────────────────────────────
    def _curve(gs, col, mean_c, std_c, n_samp, title):
        ax = fig.add_subplot(gs[0, col])
        ax.plot(timestamps, mean_c, color='#16213E', lw=1.7)
        ax.fill_between(timestamps, mean_c - std_c, mean_c + std_c,
                        color='gray', alpha=0.22, label=f'\xb11σ  N={n_samp}')
        ax.set_title(title, fontsize=8.5, fontweight='bold', pad=5)
        ax.set_xlim(t_start, t_end)
        ax.legend(fontsize=7, loc='upper left', framealpha=0.8)
        ax.grid(True, color='grey', alpha=0.22, ls='--')
        ax.tick_params(labelsize=7)
        plt.setp(ax.get_xticklabels(), visible=False)
        return ax

    # ── Helper: heatmap row ────────────────────────────────────────────────
    def _heatmap(gs, row, col, hm, title, ref_ax, vmin=None, vmax=None):
        ax = fig.add_subplot(gs[row, col], sharex=ref_ax)
        im = ax.imshow(
            hm, aspect='auto', cmap='inferno',
            vmin=vmin, vmax=vmax,
            extent=[t_start, t_end, len(hm), 0],
            interpolation='nearest',
        )
        ax.set_title(title, fontsize=7.5, fontweight='bold', pad=3)
        ax.set_ylabel('Latent rank', fontsize=7)
        ax.tick_params(labelsize=6.5)
        ax.grid(True, color='grey', alpha=0.17, ls='--')
        cax = make_axes_locatable(ax).append_axes('right', size='3%', pad=0.04)
        fig.colorbar(im, cax=cax).set_label(
            '|∂Z/∂X|', rotation=270, labelpad=8, fontsize=6.5
        )
        plt.setp(ax.get_xticklabels(), visible=False)
        return ax

    # ── Helper: 1-D summary row ────────────────────────────────────────────
    def _summary(gs, col, mean_c, sal_norm, ref_ax, fill_col, title):
        ax  = fig.add_subplot(gs[n_rows - 1, col], sharex=ref_ax)
        ax2 = ax.twinx()
        ax.plot(timestamps, mean_c, color='#16213E', lw=1.2, alpha=0.75)
        ax2.fill_between(timestamps, 0, sal_norm, color=fill_col, alpha=0.38)
        ax2.plot(timestamps, sal_norm, color=fill_col, lw=1.0, alpha=0.9)
        ax2.set_ylim(0, 2.2)
        ax2.set_yticks([])
        ax.set_xlabel('Time (min)', fontsize=7.5)
        ax.set_ylabel('Fluor.', fontsize=7, color='#16213E')
        ax.set_title(title, fontsize=7.5, fontweight='bold', pad=3)
        ax.grid(True, color='grey', alpha=0.20, ls='--')
        ax.tick_params(labelsize=6.5)
        return ax

    # ══════════════════════════════════════════════════════════════════════
    # VIEW A — correctly classified
    # ══════════════════════════════════════════════════════════════════════
    ax_a0 = _curve(gs_a, 0, mean_a, std_a, n_a,
                   f'A (Raw)  {class_name}  correct N={n_a}/{n_total_c}')
    ax_a1 = _curve(gs_a, 1, mean_a, std_a, n_a,
                   f'A (Norm)  {class_name}  correct N={n_a}/{n_total_c}')
    if is_dual:
        _heatmap(gs_a, 1, 0, hm_a_cnn_raw,  f'CNN — Raw  (top {n_cnn})',       ax_a0)
        _heatmap(gs_a, 1, 1, hm_a_cnn_norm, 'CNN — Normalised',                 ax_a1, vmin=0, vmax=1)
        _heatmap(gs_a, 2, 0, hm_a_rnn_raw,  f'{_branch} — Raw  (top {n_rnn})', ax_a0)
        _heatmap(gs_a, 2, 1, hm_a_rnn_norm, f'{_branch} — Normalised',          ax_a1, vmin=0, vmax=1)
    else:
        _heatmap(gs_a, 1, 0, hm_a_cnn_raw,  f'Raw  (top {n_cnn})', ax_a0)
        _heatmap(gs_a, 1, 1, hm_a_cnn_norm, 'Normalised',           ax_a1, vmin=0, vmax=1)
    _summary(gs_a, 0, mean_a, sal_a_raw_1d, ax_a0,
             fill_col='#1565C0', title='Σ|∂Z/∂X| (raw)')
    _summary(gs_a, 1, mean_a, sal_a_nrm_1d, ax_a1,
             fill_col='#1565C0', title='Σ|∂Z/∂X| (row-norm)')

    # ══════════════════════════════════════════════════════════════════════
    # VIEW B — class-c (or all) samples + direct input gradient
    # ══════════════════════════════════════════════════════════════════════
    scope_str = f'{class_name} only' if sample_scope == 'class' else 'all samples'
    ax_b0 = _curve(gs_b, 0, mean_b, std_b, n_b,
                   f'B (Raw)  {class_name}  {scope_str} N={n_b}')
    ax_b1 = _curve(gs_b, 1, mean_b, std_b, n_b,
                   f'B (Norm)  {class_name}  {scope_str} N={n_b}')
    if is_dual:
        _heatmap(gs_b, 1, 0, hm_b_cnn_raw,  f'CNN — Raw  (top {n_cnn})',       ax_b0)
        _heatmap(gs_b, 1, 1, hm_b_cnn_norm, 'CNN — Normalised',                 ax_b1, vmin=0, vmax=1)
        _heatmap(gs_b, 2, 0, hm_b_rnn_raw,  f'{_branch} — Raw  (top {n_rnn})', ax_b0)
        _heatmap(gs_b, 2, 1, hm_b_rnn_norm, f'{_branch} — Normalised',          ax_b1, vmin=0, vmax=1)
    else:
        _heatmap(gs_b, 1, 0, hm_b_cnn_raw,  f'Raw  (top {n_cnn})', ax_b0)
        _heatmap(gs_b, 1, 1, hm_b_cnn_norm, 'Normalised',           ax_b1, vmin=0, vmax=1)
    _summary(gs_b, 0, mean_b, sal_b_1d, ax_b0,
             fill_col='#E64A19', title='|d(cls_c)/dX|  (norm)')
    _summary(gs_b, 1, mean_b, sal_b_1d, ax_b1,
             fill_col='#E64A19', title='|d(cls_c)/dX|  (norm)')

    fig.suptitle(
        f'{class_name}  |  {art["is_type"].upper()} model  |  '
        f'A: correct N={n_a}/{n_total_c}  ◆  B: {scope_str} N={n_b}',
        fontsize=11, fontweight='bold', y=1.01,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

print('Functions defined.')


In [ ]:
# ── Multi-target View-A attribution comparison ─────────────────────────────

def plot_multi_target_view_a_norm(
    art,
    X_batch,            # (N, T, 1)
    y_batch,            # (N,) int
    y_pred,             # (N,) int
    class_indices,      # list[int] — which classes to compare
    timestamps,
    class_names=None,   # list[str]; defaults to str(idx)
    n_dims=25,
    normalise=True,     # True = row-norm heatmap collapse; False = raw heatmap collapse
    palette=None,       # list of colours; defaults to tab10
    save_path=None,
):
    """
    Overlay View-A latent attribution across multiple targets in a single figure.
    No model call needed — uses art['raw_saliency_curve'] directly.

    Each class gets one colour used for both:
      Upper panel: mean ± std fluorescence curve (correctly-classified samples only)
      Lower panel: Σ|∂Z/∂X| collapsed from the heatmap (row-norm if normalise=True)
    """
    is_dual = art['is_type'] == 'dual'
    T       = len(timestamps)
    t_step  = (timestamps[-1] - timestamps[0]) / (T - 1) if T > 1 else 1.0
    t_start = timestamps[0]  - t_step / 2
    t_end   = timestamps[-1] + t_step / 2

    if class_names is None:
        class_names = [str(c) for c in class_indices]
    if palette is None:
        _cm     = plt.get_cmap('tab10')
        palette = [_cm(i % 10) for i in range(len(class_indices))]

    n_cnn = min(n_dims, len(art['raw_saliency_curve']))
    n_rnn = min(n_dims, len(art['raw_saliency_rnn'])) if is_dual else 0

    def _norm1d(x):
        return (x - x.min()) / (x.ptp() + 1e-12)

    # ── Per-class data ─────────────────────────────────────────────────────
    records = []
    for c_idx, c_name in zip(class_indices, class_names):
        mask_a = (y_batch == c_idx) & (y_pred == c_idx)
        n_a    = int(mask_a.sum())
        n_tot  = int((y_batch == c_idx).sum())
        if n_a == 0:
            print(f'  [skip] {c_name}: no correctly-classified samples')
            continue

        hm_raw = np.array([m[mask_a].mean(0) for m in art['raw_saliency_curve'][:n_cnn]])
        sal_sum = (normalize_heatmap(hm_raw, method='row') if normalise else hm_raw).mean(0)
        if is_dual:
            hm_r    = np.array([m[mask_a].mean(0) for m in art['raw_saliency_rnn'][:n_rnn]])
            sal_sum = sal_sum + (normalize_heatmap(hm_r, method='row') if normalise else hm_r).mean(0)

        records.append({
            'name':  c_name,
            'n_a':   n_a,
            'n_tot': n_tot,
            'mean':  X_batch[mask_a, :, 0].mean(0),
            'std':   X_batch[mask_a, :, 0].std(0),
            'sal':   _norm1d(sal_sum),
        })

    if not records:
        print('Nothing to plot — all classes skipped.')
        return

    # ── Figure ─────────────────────────────────────────────────────────────
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(12, 7), sharex=True, facecolor='white',
        gridspec_kw={'height_ratios': [1.6, 1.2], 'hspace': 0.12},
    )

    for rec, color in zip(records, palette):
        lbl = f'{rec["name"]}  (N={rec["n_a"]}/{rec["n_tot"]})'
        ax_top.plot(timestamps, rec['mean'], color=color, lw=1.8, label=lbl)
        ax_top.fill_between(timestamps,
                            rec['mean'] - rec['std'],
                            rec['mean'] + rec['std'],
                            color=color, alpha=0.15)
        ax_bot.fill_between(timestamps, 0, rec['sal'], color=color, alpha=0.20)
        ax_bot.plot(timestamps, rec['sal'], color=color, lw=1.5)

    ax_top.set_ylabel('Fluorescence', fontsize=9)
    ax_top.set_xlim(t_start, t_end)
    ax_top.grid(True, color='grey', alpha=0.25, ls='--')
    ax_top.tick_params(labelsize=8)
    ax_top.legend(fontsize=8, loc='upper left', framealpha=0.85,
                  title='class  (correct N / batch N)', title_fontsize=7)

    branch   = 'CNN+GRU' if is_dual else 'CNN'
    norm_tag = 'row-norm' if normalise else 'raw'
    ax_bot.set_xlabel('Time (min)', fontsize=9)
    ax_bot.set_ylabel(f'Σ|∂Z/∂X| ({norm_tag})  [{branch}]', fontsize=9)
    ax_bot.set_ylim(0, 1)
    ax_bot.grid(True, color='grey', alpha=0.25, ls='--')
    ax_bot.tick_params(labelsize=8)

    n_dims_str = f'top {n_cnn}' + (f'+{n_rnn}' if is_dual else '') + ' dims'
    fig.suptitle(
        f'View A  —  {norm_tag} latent attribution per target  |  '
        f'{art["is_type"].upper()} model  |  {n_dims_str}\n'
        f'correctly-classified samples only',
        fontsize=10, fontweight='bold', y=1.01,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

print('Multi-target comparison function defined.')


In [ ]:
# ── Confusion matrix (full 10-fold CV) ─────────────────────────────────────

def plot_confusion_matrix(
    model_name,
    perf_data,          # Path/str to classification_performances_10fold.joblib, or pre-loaded dict
    label_names=None,
    save_path=None,
):
    """
    Confusion matrix using full 10-fold CV predictions from the joblib file.
    Color = row-normalised recall (0-1); each cell shows raw count + %.
    perf_data can be a Path to the .joblib or the already-loaded dict (avoids re-loading on repeated calls).
    """
    from sklearn.metrics import confusion_matrix as sk_cm

    if not isinstance(perf_data, dict):
        perf_data = joblib.load(perf_data)

    data = perf_data['Ori Curves']['Native'][None]

    pred_key = f'y_preds_AC_{model_name}_'
    if pred_key not in data:
        available = sorted(k for k in data if k.startswith('y_preds_AC_'))
        raise KeyError(f'{pred_key!r} not found. Available: {available}')

    y_true  = np.concatenate(data['y_trues_'])
    y_pred  = np.concatenate(data[pred_key])
    cm      = sk_cm(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    n_cls   = cm.shape[0]
    acc     = (y_pred == y_true).mean()

    if label_names is None:
        label_names = [str(i) for i in range(n_cls)]

    fig, ax = plt.subplots(figsize=(9, 8), facecolor='white')
    im      = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Recall (row-normalised)', rotation=270, labelpad=14, fontsize=9)

    for i in range(n_cls):
        for j in range(n_cls):
            v = cm_norm[i, j]
            ax.text(j, i, f'{cm[i, j]}\n({v*100:.1f}%)',
                    ha='center', va='center', fontsize=12,
                    color='white' if v > 0.55 else '#1a1a2e',
)

    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    ax.set_xticklabels(label_names, rotation=40, ha='right', fontsize=9)
    ax.set_yticklabels(label_names, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(
        f'{model_name}\n10-fold CV accuracy: {acc:.3f}  |  N={len(y_true)}',
        fontsize=11, fontweight='bold', pad=12,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

print('Confusion matrix function defined.')


In [ ]:
# ── Run: 7 figures × n_models ─────────────────────────────────────────────
#
# Toggle SAMPLE_SCOPE here or in the config cell above.
# Toggle N_DIMS to show fewer or more latent dims.

for model_name, model in models.items():
    art = non_mtl_artifacts.get(model_name)
    if art is None:
        print(f'[skip] {model_name}: no non-MTL artifact')
        continue

    print(f'\n── {model_name} ({art["is_type"]}) ──')

    # Predictions on the same batch used for artifacts
    y_pred = _get_predictions(model, X_batch)
    acc    = (y_pred == y_batch).mean()
    print(f'   batch accuracy: {acc:.3f}')

    out_dir = SAVE_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for c_idx, c_name in enumerate(LABEL_NAMES):
        n_correct = int(((y_batch == c_idx) & (y_pred == c_idx)).sum())
        n_class   = int((y_batch == c_idx).sum())
        print(f'  {c_name:15s}  correct={n_correct}/{n_class}', end='  ')
        plot_per_label_saliency_heatmap(
            art, model, X_batch, y_batch, y_pred,
            class_idx=c_idx, class_name=c_name,
            timestamps=timestamps,
            n_dims=N_DIMS, sample_scope=SAMPLE_SCOPE,
            save_path=out_dir / f'{c_name}.png',
        )
        print('saved' if n_correct > 0 else 'skipped')

print('\nDone. Outputs in:', SAVE_DIR)

---
## Optional: Toggle scope and re-run a single model × label

Change `SAMPLE_SCOPE = 'all'` below to average View B over **all** samples (not just class-c).

In [ ]:
list(models.keys())

In [ ]:
# ── Quick single-figure preview (inline, no save) ─────────────────────────
PREVIEW_MODEL  = 'cnn_gru_dual'   # must be in models dict
PREVIEW_LABEL  = LABEL_NAMES[0]   # first class in this dataset
PREVIEW_SCOPE  = 'class'          # 'class' | 'all'

_art   = non_mtl_artifacts[PREVIEW_MODEL]
_model = models[PREVIEW_MODEL]
_y_pred = _get_predictions(_model, X_batch)

for label in LABEL_NAMES:
    _c_idx  = LABEL_NAMES.index(label)
    plot_per_label_saliency_heatmap(
        _art, _model, X_batch, y_batch, _y_pred,
        class_idx=_c_idx, class_name=label,
        timestamps=timestamps,
        n_dims=N_DIMS, sample_scope=PREVIEW_SCOPE,
        save_path=None,   # inline display
    )

In [ ]:
_art  = non_mtl_artifacts[PREVIEW_MODEL]
_ypred = _get_predictions(models[PREVIEW_MODEL], X_batch)

compared_classes = LABEL_NAMES[:3]
compared_classes_idx = list(range(len(compared_classes)))

plot_multi_target_view_a_norm(
    _art, X_batch, y_batch, _ypred,
    class_indices=compared_classes_idx,
    class_names=compared_classes,
    timestamps=timestamps,
    n_dims=30,
)

In [ ]:
PERF_JOBLIB = EXP_PATH / 'classification_performances.joblib'

plot_confusion_matrix(PREVIEW_MODEL, PERF_JOBLIB, label_names=LABEL_NAMES)